# SageMaker CI/CD Pipeline
This notebook implements the CI/CD pipeline to automate training, evaluation, and deployment with checkpoints as for NYC Motor vehicle collision.

We also improve our initial model by increasing `n_estimators` to 200 and `max_depth` to 15.

In [1]:
!pip install -U "sagemaker<3.0.0"

  Using cached sagemaker-2.257.3-py3-none-any.whl.metadata (17 kB)


  Using cached attrs-25.4.0-py3-none-any.whl.metadata (10 kB)


  Using cached sagemaker_core-1.0.78-py3-none-any.whl.metadata (4.9 kB)


Using cached sagemaker-2.257.3-py3-none-any.whl (1.7 MB)
Using cached attrs-25.4.0-py3-none-any.whl (67 kB)
Using cached sagemaker_core-1.0.78-py3-none-any.whl (444 kB)


  Attempting uninstall: attrs
    Found existing installation: attrs 23.2.0


...

   ━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━ 2/3 [sagemaker]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━ 2/3 [sagemaker]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━ 2/3 [sagemaker]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━ 2/3 [sagemaker]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [sagemaker]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sagemaker-studio-analytics-extension 0.3.0 requires sparkmagic==0.22.0, but you have sparkmagic 0.21.0 which is incompatible.
sparkmagic 0.21.0 requires pandas<2.0.0,>=0.17.1, but you have pandas 2.3.3 which is incompatible.
sagemaker-train 1.13.1 requires sagemaker-core>=2.13.1, but you have sagemaker-core 1.0.78 which is incompatible.
sagemaker-serve 1.13.1 requires sagemaker-core>=2.13.1, but you have sagemaker-core 1.0.78 which is incompatible.
sagemaker-mlops 1.13.1 requires sagemaker-core>=2.13.1, but you have sagemaker-core 1.0.78 which is incompatible.


In [2]:
import os
import boto3
import sagemaker
from sagemaker.workflow.pipeline_context import PipelineSession

sess = sagemaker.Session()
region = sess.boto_region_name
role = sagemaker.get_execution_role()
pipeline_session = PipelineSession()
bucket = sess.default_bucket()
prefix = "nyc-collisions-cicd"
model_package_group_name = "NYC-Collisions-Model-Group"

print(f"Pipeline initialized in {region} for bucket {bucket}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


Pipeline initialized in us-east-1 for bucket sagemaker-us-east-1-594126158441


## 1. Define Parameters
We define parameters to execute the pipeline.

In [14]:
from sagemaker.workflow.parameters import ParameterInteger, ParameterString, ParameterFloat

processing_instance_count = ParameterInteger(name="ProcessingInstanceCount", default_value=1)
instance_type = ParameterString(name="TrainingInstanceType", default_value="ml.m5.xlarge")
model_approval_status = ParameterString(name="ModelApprovalStatus", default_value="PendingManualApproval")
# Using the raw data uploaded in Notebook 01
input_data_uri = f"s3://{bucket}/aai-540-group6/nyc-collisions/data/raw/90643364c4a54703aefd77f156a8e185.csv"
input_data = ParameterString(name="InputData", default_value=input_data_uri)
recall_threshold = ParameterFloat(name="RecallThreshold", default_value=0.20) # Require at least 20% recall for injuries

## 2. Define a Processing Step for Feature Engineering
We create a script to process raw data using our centralized modules and split it into train/test sets.

In [4]:
!mkdir -p code

In [35]:
%%writefile code/pipeline_preprocess.py
import os
import sys
import pandas as pd
from sklearn.model_selection import train_test_split

# Add src to path to import our modules from the source directory
sys.path.append('/opt/ml/processing/src')
from preprocessing import run_prep
from features import run_engineering

if __name__ == '__main__':
    base_dir = '/opt/ml/processing'
    print("Reading raw data...")
    # Read from the input channel
    input_path = f'{base_dir}/input/data/merged_sample.csv'
    if not os.path.exists(input_path):
        files = os.listdir(f'{base_dir}/input/data')
        input_path = f'{base_dir}/input/data/{files[0]}'
    
    df = pd.read_csv(input_path, low_memory=False, error_bad_lines=False, warn_bad_lines=False)

    print("Raw CSV columns:", df.columns.tolist())
    
    df.columns = df.columns.str.strip().str.lower().str.replace(r'\s+', '_', regex=True)
    
    # Print the cleaned columns
    print("Cleaned columns:", df.columns.tolist())
    
    print("Running preprocessing and feature engineering...")
    df = run_prep(df)
    df = run_engineering(df)
    
    feature_cols = ['borough', 'month', 'hour', 'is_rush_hour', 'is_weekend', 'cause_category', 'vehicle_type', 'target']
    df = df[feature_cols].dropna()
    
    print("Splitting data into training and testing sets...")
    train, test = train_test_split(df, test_size=0.2, random_state=42, stratify=df['target'])
    
    os.makedirs(f'{base_dir}/train', exist_ok=True)
    os.makedirs(f'{base_dir}/test', exist_ok=True)
    
    train.to_csv(f'{base_dir}/train/train.csv', index=False, header=True)
    test.to_csv(f'{base_dir}/test/test.csv', index=False, header=True)
    print("Preprocessing complete.")

Overwriting code/pipeline_preprocess.py


In [24]:
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.processing import ProcessingInput, ProcessingOutput
from sagemaker.workflow.steps import ProcessingStep

sklearn_processor = SKLearnProcessor(
    framework_version="1.2-1",
    instance_type="ml.m5.xlarge",
    instance_count=processing_instance_count,
    base_job_name="sklearn-nyc-process",
    role=role,
    sagemaker_session=pipeline_session,
)

processor_args = sklearn_processor.run(
    inputs=[
        ProcessingInput(source=input_data, destination="/opt/ml/processing/input/data"),
        ProcessingInput(source="../src", destination="/opt/ml/processing/src") # Mount our custom code
    ],
    outputs=[
        ProcessingOutput(output_name="train", source="/opt/ml/processing/train"),
        ProcessingOutput(output_name="test", source="/opt/ml/processing/test"),
    ],
    code="code/pipeline_preprocess.py",
)

step_process = ProcessingStep(name="NYCCrashProcess", step_args=processor_args)

INFO:sagemaker.image_uris:Defaulting to only available Python version: py3


## 3. Define a Training Step
We train our Random Forest model using the **improved hyperparameters** (`n_estimators`=200, `max_depth`=15).

In [25]:
from sagemaker.estimator import Estimator
from sagemaker.inputs import TrainingInput
from sagemaker.workflow.steps import TrainingStep

image_uri = sagemaker.image_uris.retrieve(
    framework="sklearn",
    region=region,
    version="1.2-1"
)

# Use our existing sagemaker_train.py entry point
rf_train = Estimator(
    image_uri=image_uri,
    entry_point="sagemaker_train.py",
    source_dir="../src",
    instance_type=instance_type,
    instance_count=1,
    output_path=f"s3://{bucket}/{prefix}/model",
    role=role,
    sagemaker_session=pipeline_session,
    hyperparameters={"n-estimators": 200, "max-depth": 15} # Improved parameters for CI/CD
)

train_args = rf_train.fit(
    inputs={
        "train": TrainingInput(
            s3_data=step_process.properties.ProcessingOutputConfig.Outputs["train"].S3Output.S3Uri,
            content_type="text/csv",
        )
    }
)

step_train = TrainingStep(
    name="NYCCrashTrain",
    step_args=train_args,
)

INFO:sagemaker.image_uris:Defaulting to only available Python version: py3


INFO:sagemaker.image_uris:Defaulting to only supported image scope: cpu.


INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.


## 4. Define an Evaluation Step
Evaluates the trained model purely on the holdout test data.

In [26]:
%%writefile code/pipeline_evaluate.py
import os
import json
import tarfile
import joblib
import pandas as pd
from sklearn.metrics import recall_score, accuracy_score, f1_score

if __name__ == "__main__":
    model_path = "/opt/ml/processing/model/model.tar.gz"
    with tarfile.open(model_path) as tar:
        tar.extractall(path=".")
    
    model = joblib.load("model.joblib")
    test_df = pd.read_csv("/opt/ml/processing/test/test.csv")
    
    X_test = test_df.drop("target", axis=1)
    y_test = test_df["target"]
    
    preds = model.predict(X_test)
    
    recall = recall_score(y_test, preds)
    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds)
    
    report_dict = {
        "classification_metrics": {
            "recall": {"value": recall, "standard_deviation": "NaN"},
            "accuracy": {"value": acc, "standard_deviation": "NaN"},
            "f1": {"value": f1, "standard_deviation": "NaN"}
        }
    }
    
    output_dir = "/opt/ml/processing/evaluation"
    os.makedirs(output_dir, exist_ok=True)
    with open(f"{output_dir}/evaluation.json", "w") as f:
        f.write(json.dumps(report_dict))
    
    print(f"Evaluation complete. Recall: {recall:.4f}, Accuracy: {acc:.4f}, F1: {f1:.4f}")

Overwriting code/pipeline_evaluate.py


In [27]:
from sagemaker.processing import ScriptProcessor
from sagemaker.workflow.properties import PropertyFile

script_eval = ScriptProcessor(
    image_uri=image_uri,
    command=["python3"],
    instance_type="ml.m5.xlarge",
    instance_count=1,
    base_job_name="script-nyc-eval",
    role=role,
    sagemaker_session=pipeline_session,
)

eval_args = script_eval.run(
    inputs=[
        ProcessingInput(
            source=step_train.properties.ModelArtifacts.S3ModelArtifacts,
            destination="/opt/ml/processing/model",
        ),
        ProcessingInput(
            source=step_process.properties.ProcessingOutputConfig.Outputs["test"].S3Output.S3Uri,
            destination="/opt/ml/processing/test",
        ),
    ],
    outputs=[
        ProcessingOutput(output_name="evaluation", source="/opt/ml/processing/evaluation"),
    ],
    code="code/pipeline_evaluate.py",
)

evaluation_report = PropertyFile(
    name="EvaluationReport", output_name="evaluation", path="evaluation.json"
)

step_eval = ProcessingStep(
    name="NYCCrashEval",
    step_args=eval_args,
    property_files=[evaluation_report],
)

## 5. Register Model Step & Checkpoint Conditions
We implement the CI/CD checkpoint here. We only register the model to the Model Registry (our deployment artifact) IF the `recall` meets the required threshold.

In [28]:
from sagemaker.model import Model
from sagemaker.model_metrics import MetricsSource, ModelMetrics
from sagemaker.workflow.model_step import ModelStep
from sagemaker.workflow.conditions import ConditionGreaterThanOrEqualTo
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.functions import JsonGet
from sagemaker.workflow.fail_step import FailStep

# Define Model for Registration
model = Model(
    image_uri=image_uri,
    model_data=step_train.properties.ModelArtifacts.S3ModelArtifacts,
    sagemaker_session=pipeline_session,
    role=role,
    entry_point='sagemaker_train.py',
    source_dir='../src'
)

# Register Step
model_metrics = ModelMetrics(
    model_statistics=MetricsSource(
        s3_uri="{}/evaluation.json".format(
            step_eval.arguments["ProcessingOutputConfig"]["Outputs"][0]["S3Output"]["S3Uri"]
        ),
        content_type="application/json",
    )
)

register_args = model.register(
    content_types=["text/csv"],
    response_types=["text/csv"],
    inference_instances=["ml.m5.large", "ml.m5.xlarge"],
    transform_instances=["ml.m5.xlarge"],
    model_package_group_name=model_package_group_name,
    approval_status=model_approval_status,
    model_metrics=model_metrics,
)
step_register = ModelStep(name="NYCCrashRegisterModel", step_args=register_args)

# Fail Step
step_fail = FailStep(
    name="NYCCrashRecallFail",
    error_message="Execution failed due to Recall not meeting the required threshold.",
)

# Condition Step (The CI/CD Checkpoint)
cond_gte = ConditionGreaterThanOrEqualTo(
    left=JsonGet(
        step_name=step_eval.name,
        property_file=evaluation_report,
        json_path="classification_metrics.recall.value",
    ),
    right=recall_threshold,
)

step_cond = ConditionStep(
    name="NYCCrashRecallCond",
    conditions=[cond_gte],
    if_steps=[step_register],
    else_steps=[step_fail],
)

## 6. Build and Execute the Pipeline

In [36]:
from sagemaker.workflow.pipeline import Pipeline

pipeline_name = f"NYCCrashPipeline"
pipeline = Pipeline(
    name=pipeline_name,
    parameters=[
        processing_instance_count,
        instance_type,
        model_approval_status,
        input_data,
        recall_threshold,
    ],
    steps=[step_process, step_train, step_eval, step_cond],
)

# Submit pipeline to SageMaker
pipeline.upsert(role_arn=role)

# Start execution
execution = pipeline.start()
print(f"Started pipeline execution: {execution.arn}")

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.


INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.


Started pipeline execution: arn:aws:sagemaker:us-east-1:594126158441:pipeline/NYCCrashPipeline/execution/fc5w5eo7wnm7


In [37]:
# Wait for execution to complete (can take ~15 minutes)
execution.wait()

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:2                                                                                    │
│                                                                                                  │
│   1 # Wait for execution to complete (can take ~15 minutes)                                      │
│ ❱ 2 execution.wait()                                                                             │
│   3                                                                                              │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline.py:998 in wait               │
│                                                                                                  │
│    995 │   │   waiter = botocore.waiter.create_waiter_with_client(                               │
│    996 │   │   │   waiter_id, model, self.sagemaker_session.sagemaker_client                     │
│    997 │   │   )                                                                                 │
│ ❱  998 │   │   waiter.wait(PipelineExecutionArn=self.arn)                                        │
│    999 │                                                                                         │
│   1000 │   def result(self, step_name: str):                                                     │
│   1001 │   │   """Retrieves the output of the provided step if it is a ``@step`` decorated func  │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/botocore/waiter.py:58 in wait                            │
│                                                                                                  │
│    55 │   # Waiter.wait method. This is needed to attach a docstring to the                      │
│    56 │   # method.                                                                              │
│    57 │   def wait(self, **kwargs):                                                              │
│ ❱  58 │   │   Waiter.wait(self, **kwargs)                                                        │
│    59 │                                                                                          │
│    60 │   wait.__doc__ = WaiterDocstring(                                                        │
│    61 │   │   waiter_name=waiter_name,                                                           │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/botocore/context.py:123 in wrapper                       │
│                                                                                                  │
│   120 │   │   │   with start_as_current_context():                                               │
│   121 │   │   │   │   if hook:                                                                   │
│   122 │   │   │   │   │   hook()                                                                 │
│ ❱ 123 │   │   │   │   return func(*args, **kwargs)                                               │
│   124 │   │                                                                                      │
│   125 │   │   return wrapper                                                                     │
│   126                                                                                            │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/botocore/waiter.py:378 in wait                           │
│                                                                                                  │
│   375 │   │   │   │   return                               

In [ ]:
execution.list_steps()